# Quafu cloud DTC-QRC acquisition — Baihua / 02_q6_t6_l3_batch02

This notebook follows the structure and Quafu API call pattern of the supplied
`02_qrc_quafu_Baihua12_T12_modified.ipynb`, with only the manuscript protocol,
NPZ path, physical chain, task naming, and randomized ordering updated.

It uses Q=T=6, g=0.84, W=0.60, reservoir seed 0, and the task-specific L/fold
configuration. The cloud saves job manifests and raw task results only.


## Cloud-side requirements

The notebook follows the supplied Quafu template and uses only `numpy` and
`quarkstudio`/`quark`. The API credential is held in the environment variable
`QPU_API_TOKEN`. The first configuration cell prompts for it with `getpass`
when the variable is absent; the value is not saved into the notebook.

The cloud stage constructs QLISP circuits, calls `Task.run`, calls
`Task.result`, and saves raw result JSON files. Token generation, observable
reconstruction, model training, AUC calculation, and plotting remain local.


In [ ]:
# ===============================
# 1) Configuration
# ===============================
CONFIG = {
    "run": {
        # Choose: "dry_build", "submit_qrc_jobs", "collect_qrc_results".
        "mode": "submit_qrc_jobs",
        "molecule_limit": None,
        "max_submit_per_run": None,
        "overwrite_jobs": False,
        "overwrite_results": False,
        "poll_sleep_sec": 0.0,
    },
    "tokens": {
        "max_events": 6,
        "token_dim": 18,
        "use_tanh_before_encoding": True,
    },
    "qrc": {
        "num_qubits": 6,
        "features_per_qubit": 3,
        "g": 0.84,
        "W_scale": 0.60,
        "layers_per_injection": 3,
        "fold_factor": 1,
        "reservoir_seed": 0,
        "J_low": -0.9,
        "J_high": 0.9,
        "chip": 'Baihua',
        "repeat": 1,
        "mapping_mode": "physical_direct",
        "backend_num_qubits": 156,
        "physical_qubits": [72, 73, 74, 75, 76, 77],
        "compile": True,
        "compiler": "quarkcircuit",
        "correct": False,
        "open_dd": None,
        "task_name_prefix": 'baihua_02_q6_t6_l3_batch02',
        "full_bit_order": "rightmost_is_c0",
        "compact_bit_order": "rightmost_is_first_measured",
    },
    "submission": {
        "order_seed": 31002,
        "label_blind_random_order": True,
        "interleaved_l1_pair": False,
    },
    "artifacts": {
        "token_file": 'bace_bal400_canonical_v3_q6_t6__02_q6_t6_l3_batch02.npz',
        "save_dir": 'baihua_02_q6_t6_l3_batch02_cloud_results',
        "job_manifest_path": 'baihua_02_q6_t6_l3_batch02_cloud_results/baihua_02_q6_t6_l3_batch02_job_manifest.jsonl',
        "raw_result_dir": 'baihua_02_q6_t6_l3_batch02_cloud_results/raw_counts_baihua_02_q6_t6_l3_batch02',
        "download_zip_name": 'baihua_02_q6_t6_l3_batch02_cloud_results.zip',
    },
}

# The supplied template reads QPU_API_TOKEN from the environment.
import os
from getpass import getpass
if not os.getenv("QPU_API_TOKEN", ""):
    os.environ["QPU_API_TOKEN"] = getpass("QPU_API_TOKEN: ")

PROTOCOL_VERSION = '2026-07-28-mainline-canonical-v3-q6t6-five-task-strict-template-v4'
BACKEND_KEY = 'baihua'
CHIP = CONFIG["qrc"]["chip"]
BACKEND_NUM_QUBITS = CONFIG["qrc"]["backend_num_qubits"]
PHYSICAL_QUBITS = CONFIG["qrc"]["physical_qubits"]
CALIBRATED_AT = '2026-07-28 19:23:48'
CURRENT_TASK_KEY = '02_q6_t6_l3_batch02'
CURRENT_TASK_FOLDER = '02_q6_t6_l3_batch02'
CURRENT_TASK = {'key': '02_q6_t6_l3_batch02', 'folder': '02_q6_t6_l3_batch02', 'layers': 3, 'fold_factor': 1, 'order_seed': 31002, 'role': 'independent L3 hardware batch 2', 'pair_group': None}
SUBMISSION_TASKS = [{'key': '02_q6_t6_l3_batch02', 'folder': '02_q6_t6_l3_batch02', 'layers': 3, 'fold_factor': 1, 'order_seed': 31002, 'role': 'independent L3 hardware batch 2', 'pair_group': None}]
IS_INTERLEAVED_L1_PAIR = False
Q = CONFIG["qrc"]["num_qubits"]
T = CONFIG["tokens"]["max_events"]
G = CONFIG["qrc"]["g"]
W = CONFIG["qrc"]["W_scale"]
RESERVOIR_SEED = CONFIG["qrc"]["reservoir_seed"]
REPEAT = CONFIG["qrc"]["repeat"]
DATASET_FILE = CONFIG["artifacts"]["token_file"]
EXPECTED_DATASET_SHA256 = '00b3fdbab64cb95f61b8580bd7e975ee9f1c9d7df1b44ce60ebe61a643a3eb04'
EXPECTED_CURRENT_CX = 180


In [ ]:
import hashlib
import json
import math
import re
import shutil
import time
from pathlib import Path

import numpy as np


def locate_notebook_dir():
    direct = Path.cwd()
    if (direct / DATASET_FILE).is_file():
        return direct
    matches = list(direct.glob(f"**/{DATASET_FILE}"))
    if len(matches) == 1:
        return matches[0].parent
    raise FileNotFoundError(
        f"Cannot locate {DATASET_FILE}; run from the extracted five-task package."
    )


NOTEBOOK_DIR = locate_notebook_dir()
PACKAGE_DIR = NOTEBOOK_DIR.parent
TASK_BY_KEY = {value["key"]: dict(value) for value in SUBMISSION_TASKS}


def task_paths(task_key):
    spec = TASK_BY_KEY.get(task_key)
    if spec is None and task_key == CURRENT_TASK_KEY:
        spec = dict(CURRENT_TASK)
    if spec is None:
        raise KeyError(task_key)
    task_dir = PACKAGE_DIR / spec["folder"]
    stem = f"{BACKEND_KEY}_{task_key}"
    root = task_dir / f"{stem}_cloud_results"
    return {
        "task_dir": task_dir,
        "root": root,
        "manifest": root / f"{stem}_job_manifest.jsonl",
        "raw_dir": root / f"raw_counts_{stem}",
        "order": root / f"{stem}_submission_order.json",
        "summary": root / f"{stem}_collection_summary.json",
        "zip": task_dir / f"{stem}_cloud_results.zip",
    }


CURRENT_PATHS = task_paths(CURRENT_TASK_KEY)


def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def json_default(value):
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, Path):
        return str(value)
    return str(value)


def scalar_text(value):
    array = np.asarray(value)
    return str(array.item()) if array.shape == () else str(value)


In [ ]:
def load_processed_dataset():
    path = NOTEBOOK_DIR / DATASET_FILE
    actual_hash = sha256_file(path)
    if actual_hash != EXPECTED_DATASET_SHA256:
        raise AssertionError(
            f"Dataset SHA-256 mismatch: {actual_hash} != {EXPECTED_DATASET_SHA256}"
        )
    with np.load(path, allow_pickle=False) as package:
        bundle = {
            "indices": package["indices"].astype(np.int64),
            "tokens": package["tokens"].astype(np.float32),
            "token_hashes": package["token_hashes"].astype(str),
            "schema": scalar_text(package["token_schema_version"]),
            "metadata": json.loads(scalar_text(package["metadata_json"])),
        }
    if bundle["tokens"].shape != (400, T, 3 * Q):
        raise AssertionError(bundle["tokens"].shape)
    if bundle["schema"] != "local-bfs-canonical-isomeric-reparse-stable-v3":
        raise AssertionError(bundle["schema"])
    if bundle["metadata"].get("global_descriptors") is not False:
        raise AssertionError("Global descriptors are forbidden")
    if bundle["metadata"].get("task_guided_traversal") is not False:
        raise AssertionError("Task-guided traversal is forbidden")
    return bundle


BUNDLE = load_processed_dataset()
print(
    {
        "backend": CHIP,
        "task": CURRENT_TASK_KEY,
        "dataset_shape": BUNDLE["tokens"].shape,
        "physical_chain": PHYSICAL_QUBITS,
        "calibration_timestamp": CALIBRATED_AT,
        "submission_circuit_count": 800 if IS_INTERLEAVED_L1_PAIR else 400,
        "collection_circuit_count": 400,
    }
)


In [ ]:
def norm_edge(left, right):
    left, right = int(left), int(right)
    return (left, right) if left < right else (right, left)


def baihua_edges():
    edges = set()
    rows, cols = 12, 13
    for row in range(rows):
        for col in range(cols - 1):
            edges.add(norm_edge(row * cols + col, row * cols + col + 1))
    for row in range(rows - 1):
        vertical_cols = (2, 6, 10) if row % 2 == 0 else (0, 4, 8, 12)
        for col in vertical_cols:
            edges.add(norm_edge(row * cols + col, (row + 1) * cols + col))
    return edges


def shenglian_edges():
    edges = set()
    rows, cols = 12, 7
    for row in range(rows - 1):
        for col in range(cols):
            upper = row * cols + col
            lower_same = (row + 1) * cols + col
            edges.add(norm_edge(upper, lower_same))
            if row % 2 == 0 and col > 0:
                edges.add(norm_edge(upper, (row + 1) * cols + col - 1))
    return edges


def validate_chain():
    if len(PHYSICAL_QUBITS) != Q or len(set(PHYSICAL_QUBITS)) != Q:
        raise AssertionError("Physical chain must contain Q unique qubits")
    if min(PHYSICAL_QUBITS) < 0 or max(PHYSICAL_QUBITS) >= BACKEND_NUM_QUBITS:
        raise AssertionError("Physical chain is outside the backend register")
    coupling = baihua_edges() if BACKEND_KEY == "baihua" else shenglian_edges()
    missing = [
        norm_edge(left, right)
        for left, right in zip(PHYSICAL_QUBITS[:-1], PHYSICAL_QUBITS[1:])
        if norm_edge(left, right) not in coupling
    ]
    if missing:
        raise AssertionError(f"Nonconnected physical chain; missing={missing}")


def dtc_parameters():
    rng = np.random.default_rng(RESERVOIR_SEED)
    return {
        "J": rng.uniform(-0.9, 0.9, size=Q - 1).astype(np.float32),
        "h": rng.uniform(-W, W, size=Q).astype(np.float32),
    }


def insert_barrier(qc):
    barrier = getattr(qc, "barrier", None)
    if barrier is None:
        raise RuntimeError(
            "fold3 requires QuantumCircuit.barrier so B B^dagger B cannot "
            "be silently cancelled before platform compilation."
        )
    try:
        barrier(*[int(value) for value in PHYSICAL_QUBITS])
    except TypeError:
        barrier([int(value) for value in PHYSICAL_QUBITS])


def rzz_block(qc, left, right, angle):
    qc.cx(int(left), int(right))
    qc.rz(float(angle), int(right))
    qc.cx(int(left), int(right))


def folded_rzz(qc, left, right, angle, fold_factor):
    if int(fold_factor) == 1:
        rzz_block(qc, left, right, angle)
        return
    if int(fold_factor) != 3:
        raise ValueError(fold_factor)
    rzz_block(qc, left, right, angle)
    insert_barrier(qc)
    rzz_block(qc, left, right, -float(angle))
    insert_barrier(qc)
    rzz_block(qc, left, right, angle)


def build_circuit(token_sequence, task_spec=None):
    from quark.circuit import QuantumCircuit

    spec = dict(CURRENT_TASK if task_spec is None else task_spec)
    layers = int(spec["layers"])
    fold_factor = int(spec["fold_factor"])
    validate_chain()
    sequence = np.tanh(np.asarray(token_sequence, dtype=np.float32))
    if sequence.shape != (T, 3 * Q):
        raise AssertionError(sequence.shape)
    params = dtc_parameters()
    qc = QuantumCircuit(BACKEND_NUM_QUBITS, BACKEND_NUM_QUBITS)
    for event in range(T):
        values = sequence[event].reshape(Q, 3)
        for logical, physical in enumerate(PHYSICAL_QUBITS):
            theta, phi, lam = values[logical]
            qc.u(float(theta), float(phi), float(lam), int(physical))
        for _ in range(layers):
            for physical in PHYSICAL_QUBITS:
                qc.rx(G * math.pi, int(physical))
            for start in (0, 1):
                for logical in range(start, Q - 1, 2):
                    folded_rzz(
                        qc,
                        PHYSICAL_QUBITS[logical],
                        PHYSICAL_QUBITS[logical + 1],
                        float(params["J"][logical]),
                        fold_factor,
                    )
            for logical, physical in enumerate(PHYSICAL_QUBITS):
                qc.rz(float(params["h"][logical]) * math.pi, int(physical))
    qc.measure(PHYSICAL_QUBITS, PHYSICAL_QUBITS)
    expected_cx = 2 * T * layers * (Q - 1) * fold_factor
    metadata = {
        "q": Q,
        "t": T,
        "layers": layers,
        "fold_factor": fold_factor,
        "g": G,
        "W": W,
        "reservoir_seed": RESERVOIR_SEED,
        "J": params["J"].tolist(),
        "h": params["h"].tolist(),
        "interaction": "CX-RZ(J_i)-CX",
        "folding": "B B^dagger B with barriers" if fold_factor == 3 else "none",
        "circuit_qubits": list(PHYSICAL_QUBITS),
        "measured_clbits": list(PHYSICAL_QUBITS),
        "classical_num_bits": BACKEND_NUM_QUBITS,
        "nominal_two_qubit_gates": expected_cx,
    }
    if expected_cx > 200:
        raise AssertionError(f"Two-qubit gate cap exceeded: {expected_cx}")
    return qc, metadata


validate_chain()
preview_circuit, preview_metadata = build_circuit(BUNDLE["tokens"][0])
if int(preview_metadata["nominal_two_qubit_gates"]) != int(EXPECTED_CURRENT_CX):
    raise AssertionError(
        (
            preview_metadata["nominal_two_qubit_gates"],
            EXPECTED_CURRENT_CX,
        )
    )
print(
    f"Topology/circuit validation passed; current nominal CX count = "
    f"{EXPECTED_CURRENT_CX}"
)
print("QLISP preview (format is backend-version dependent):")
preview_qlisp = preview_circuit.to_qlisp
if isinstance(preview_qlisp, (list, tuple)):
    for item in list(preview_qlisp)[:20]:
        print(item)
    print(f"Total QLISP items: {len(preview_qlisp)}")
else:
    print(str(preview_qlisp)[:2000])
    print(f"QLISP object type: {type(preview_qlisp)}")


In [ ]:
def read_jsonl(path):
    path = Path(path)
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]


def atomic_write_jsonl(records, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for record in records:
            handle.write(
                json.dumps(record, ensure_ascii=False, default=json_default) + "\n"
            )
    temporary.replace(path)


def init_quafu_task_manager():
    token = os.getenv("QPU_API_TOKEN", "")
    if not token:
        raise RuntimeError(
            "QPU_API_TOKEN is empty. Enter it in the configuration cell."
        )
    from quark import Task
    return Task(token)


def task_manager():
    # Compatibility alias; submission/collection use the supplied template API.
    return init_quafu_task_manager()


def randomized_specs():
    seed = int(SUBMISSION_TASKS[0]["order_seed"])
    if any(int(value["order_seed"]) != seed for value in SUBMISSION_TASKS):
        raise AssertionError("Coordinated submission tasks must share one seed")
    rng = np.random.default_rng(seed)
    rows = rng.permutation(len(BUNDLE["indices"])).astype(int)
    molecule_limit = CONFIG.get("run", {}).get("molecule_limit", None)
    if molecule_limit is not None:
        rows = rows[: int(molecule_limit)]
    specs = []
    for molecule_rank, row in enumerate(rows):
        local_tasks = [dict(value) for value in SUBMISSION_TASKS]
        if IS_INTERLEAVED_L1_PAIR:
            rng.shuffle(local_tasks)
        for within_pair_rank, task_spec in enumerate(local_tasks):
            specs.append(
                {
                    "row": int(row),
                    "submission_rank": int(molecule_rank),
                    "global_submission_rank": int(len(specs)),
                    "within_pair_rank": int(within_pair_rank),
                    "original_index": int(BUNDLE["indices"][row]),
                    "task_key": task_spec["key"],
                }
            )
    for task_spec in SUBMISSION_TASKS:
        paths = task_paths(task_spec["key"])
        paths["root"].mkdir(parents=True, exist_ok=True)
        selected = [value for value in specs if value["task_key"] == task_spec["key"]]
        paths["order"].write_text(
            json.dumps(
                {
                    "protocol_version": PROTOCOL_VERSION,
                    "backend": CHIP,
                    "task_key": task_spec["key"],
                    "seed": seed,
                    "label_blind_random_permutation": True,
                    "interleaved_pair": IS_INTERLEAVED_L1_PAIR,
                    "records": selected,
                },
                indent=2,
            )
            + "\n",
            encoding="utf-8",
        )
    return specs


def submit_all_randomized():
    tmgr = init_quafu_task_manager()
    specs = randomized_specs()
    records_by_task = {}
    by_index_by_task = {}
    for task_spec in SUBMISSION_TASKS:
        key = task_spec["key"]
        records = read_jsonl(task_paths(key)["manifest"])
        records_by_task[key] = records
        by_index_by_task[key] = {
            int(record["original_index"]): record for record in records
        }
    submitted = 0
    max_submit = CONFIG.get("run", {}).get("max_submit_per_run", None)
    max_submit = None if max_submit is None else int(max_submit)
    overwrite = bool(CONFIG.get("run", {}).get("overwrite_jobs", False))
    poll_sleep = float(CONFIG.get("run", {}).get("poll_sleep_sec", 0.0))
    for spec in specs:
        if max_submit is not None and submitted >= max_submit:
            break
        task_key = spec["task_key"]
        old = by_index_by_task[task_key].get(int(spec["original_index"]))
        if old is not None and old.get("tid") is not None and not overwrite:
            continue
        task_spec = next(
            value for value in SUBMISSION_TASKS if value["key"] == task_key
        )
        row = int(spec["row"])
        qc, metadata = build_circuit(BUNDLE["tokens"][row], task_spec)
        circuit_id = (
            f"{BACKEND_KEY}_{task_key}_grank{spec['global_submission_rank']:05d}"
            f"_id{int(spec['original_index']):05d}"
        )
        options = {
            "compiler": "quarkcircuit",
            "correct": False,
            "target_qubits": list(PHYSICAL_QUBITS),
        }
        task_payload = {
            "chip": CHIP,
            "name": circuit_id,
            "circuit": qc.to_qlisp,
            "compile": True,
            "correct": False,
            "options": options,
        }
        tid = tmgr.run(task_payload, repeat=REPEAT)
        record = {
            **spec,
            **metadata,
            "protocol_version": PROTOCOL_VERSION,
            "backend_key": BACKEND_KEY,
            "chip": CHIP,
            "physical_qubits": list(PHYSICAL_QUBITS),
            "calibration_timestamp_from_image": CALIBRATED_AT,
            "dataset_file": (
                f"bace_bal400_canonical_v3_q6_t6__{task_spec['key']}.npz"
            ),
            "dataset_sha256": EXPECTED_DATASET_SHA256,
            "token_hash": str(BUNDLE["token_hashes"][row]),
            "circuit_id": circuit_id,
            "tid": str(tid),
            "repeat": REPEAT,
            "status": "Submitted",
            "compile": True,
            "task_options": options,
        }
        by_index_by_task[task_key][int(spec["original_index"])] = record
        ordered = sorted(
            by_index_by_task[task_key].values(),
            key=lambda value: int(value["submission_rank"]),
        )
        atomic_write_jsonl(ordered, task_paths(task_key)["manifest"])
        submitted += 1
        print(
            f"[{submitted:04d}] global_rank={spec['global_submission_rank']:04d} "
            f"task={task_key} id={spec['original_index']} tid={tid}"
        )
        if poll_sleep > 0:
            time.sleep(poll_sleep)
    manifests = {
        value["key"]: str(task_paths(value["key"])["manifest"])
        for value in SUBMISSION_TASKS
    }
    print(
        f"Submitted {submitted} new jobs; coordinated total expected="
        f"{800 if IS_INTERLEAVED_L1_PAIR else 400}"
    )
    return manifests


def extract_counts(result):
    if not isinstance(result, dict):
        return None
    for key in ("count", "counts"):
        if result.get(key):
            return result[key]
    for parent in ("data", "result"):
        child = result.get(parent)
        if isinstance(child, dict):
            for key in ("count", "counts"):
                if child.get(key):
                    return child[key]
    return None


def refresh_download_zip():
    paths = CURRENT_PATHS
    archive_base = str(paths["zip"].with_suffix(""))
    archive_path = Path(
        shutil.make_archive(
            archive_base,
            "zip",
            root_dir=paths["root"].parent,
            base_dir=paths["root"].name,
        )
    )
    print(f"Download this file: {archive_path.resolve()}")
    return archive_path


def collect_raw_results():
    paths = CURRENT_PATHS
    records = read_jsonl(paths["manifest"])
    if len(records) != 400:
        raise RuntimeError(
            f"Expected 400 submitted records for {CURRENT_TASK_KEY}, got "
            f"{len(records)}. Run ACTION='submit' first."
        )
    tmgr = init_quafu_task_manager()
    paths["raw_dir"].mkdir(parents=True, exist_ok=True)
    completed = 0
    pending = 0
    failed = 0
    for position, record in enumerate(
        sorted(records, key=lambda value: int(value["submission_rank"]))
    ):
        raw_name = record.get(
            "raw_file_name",
            f"rank{int(record['submission_rank']):05d}"
            f"_id{int(record['original_index']):05d}"
            f"_tid{record['tid']}.json",
        )
        raw_path = paths["raw_dir"] / raw_name
        result = None
        if raw_path.is_file():
            try:
                result = json.loads(raw_path.read_text(encoding="utf-8"))
            except Exception:
                result = None
        if result is None or not extract_counts(result):
            try:
                result = tmgr.result(record["tid"])
            except Exception as error:
                record["status"] = f"FetchFailed: {error}"
                failed += 1
                atomic_write_jsonl(records, paths["manifest"])
                print(record["circuit_id"], record["status"])
                continue
            raw_path.write_text(
                json.dumps(result, ensure_ascii=False, indent=2, default=json_default),
                encoding="utf-8",
            )
        counts = extract_counts(result)
        record["raw_file_name"] = raw_name
        record["raw_result_path_cloud"] = str(raw_path)
        if counts:
            record["status"] = "CountsDownloaded"
            record["shots_downloaded"] = int(
                round(sum(float(value) for value in counts.values()))
            )
            completed += 1
        else:
            record["status"] = (
                (
                    result.get("status")
                    or result.get("task_status")
                    or result.get("state")
                    or "PendingNoCounts"
                )
                if isinstance(result, dict)
                else "PendingNoCounts"
            )
            pending += 1
        atomic_write_jsonl(records, paths["manifest"])
        print(
            f"[{position + 1:03d}/400] task={CURRENT_TASK_KEY} "
            f"rank={record['submission_rank']} status={record['status']}"
        )
    summary = {
        "protocol_version": PROTOCOL_VERSION,
        "backend": CHIP,
        "task_key": CURRENT_TASK_KEY,
        "physical_qubits": list(PHYSICAL_QUBITS),
        "submitted_records": len(records),
        "counts_downloaded": completed,
        "pending_no_counts": pending,
        "fetch_failed": failed,
        "complete": completed == 400,
        "cloud_processing": (
            "raw result retrieval only; no label loading, feature extraction, "
            "training, AUC, or plotting"
        ),
    }
    paths["summary"].write_text(
        json.dumps(summary, indent=2) + "\n", encoding="utf-8"
    )
    archive = refresh_download_zip()
    print(json.dumps(summary, indent=2))
    return archive



# Supplied-template stage names.
def submit_qrc_jobs(config=CONFIG):
    return submit_all_randomized()


def collect_qrc_results(config=CONFIG):
    return collect_raw_results()


In [ ]:
# ===============================
# 7) Execute configured cloud stage
# ===============================
def execute_configured_stage(config: dict):
    mode = config.get("run", {}).get("mode", "dry_build")
    if mode == "dry_build":
        return preview_qlisp
    if mode == "submit_qrc_jobs":
        return submit_qrc_jobs(config)
    if mode == "collect_qrc_results":
        return collect_qrc_results(config)
    raise ValueError(f"Unknown mode: {mode}")


stage_output = execute_configured_stage(CONFIG)
stage_output
